In [2]:
#!/usr/bin/env python3
import os
import tarfile
import glob
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from collections import defaultdict
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

In [3]:
base_dir = "/Users/tereza/spring_2025/STAT_4830/STAT-4830-GOALZ-project/data"
tar_file = os.path.join(base_dir, "c22_features.tar.gz")
output_dir = os.path.join(base_dir, "catch22_feats_sleepedf")
os.makedirs(output_dir, exist_ok=True)

print("Step 1: Extracting tarfile...")
if len(os.listdir(output_dir)) == 0:
    with tarfile.open(tar_file, "r:gz") as tar:
        for member in tar.getmembers():
            member_name = os.path.basename(member.name)
            if member.isreg():
                f = tar.extractfile(member)
                if f is not None:
                    with open(os.path.join(output_dir, member_name), 'wb') as outfile:
                        outfile.write(f.read())
print("Extraction complete!")

csv_files = glob.glob(os.path.join(output_dir, "*.csv"))
npz_files = glob.glob(os.path.join(output_dir, "*.npz"))
print(f"Found {len(csv_files)} CSV files and {len(npz_files)} NPZ files")

mapping_file = os.path.join(output_dir, "subject_recording_mapping.csv")
if os.path.exists(mapping_file):
    subject_mapping = pd.read_csv(mapping_file)
    print(f"Found mapping file with {len(subject_mapping)} recordings")
    print(f"Number of unique subjects: {subject_mapping['subject_id'].nunique()}")
    
    subject_counts = subject_mapping['subject_id'].value_counts()
    print("\nDistribution of recordings per subject:")
    print(f"Min: {subject_counts.min()}, Max: {subject_counts.max()}, Mean: {subject_counts.mean():.2f}")
    plt.figure(figsize=(10, 6))
    plt.hist(subject_counts, bins=range(1, subject_counts.max() + 2), alpha=0.7)
    plt.title('Recordings per Subject')
    plt.xlabel('Number of Recordings')
    plt.ylabel('Count')
    plt.grid(alpha=0.3)
    plt.savefig('recordings_per_subject.png')
    plt.close()

if not os.path.exists(mapping_file):
    recordings = []
    subject_ids = []
    
    for file in csv_files:
        rec_id = os.path.basename(file).replace("_catch22_features.csv", "")
        recordings.append(rec_id)
        
        if rec_id.startswith('SC4'):
            subject_ids.append(rec_id[:5])
        elif rec_id.startswith('ST7'):
            subject_ids.append(rec_id[:5])
        else:
            subject_ids.append(rec_id[:6])
    
    subject_mapping = pd.DataFrame({
        'recording_id': recordings,
        'subject_id': subject_ids
    })
    
    print(f"Created mapping with {len(subject_mapping)} recordings")
    print(f"Number of unique subjects: {subject_mapping['subject_id'].nunique()}")

sc_subjects = subject_mapping[subject_mapping['subject_id'].str.startswith('SC')]['subject_id'].nunique()
st_subjects = subject_mapping[subject_mapping['subject_id'].str.startswith('ST')]['subject_id'].nunique()
print(f"\nSubject distribution:")
print(f"SC (Sleep Cassette) subjects: {sc_subjects}")
print(f"ST (Sleep Telemetry) subjects: {st_subjects}")

sample_file = csv_files[0]
sample_data = pd.read_csv(sample_file)
print(f"\nSample data structure from {os.path.basename(sample_file)}:")
print(f"Shape: {sample_data.shape}")
print(f"Columns: {len(sample_data.columns)}")

eeg_cols = [col for col in sample_data.columns if col.startswith('eeg_')]
eog_cols = [col for col in sample_data.columns if col.startswith('eog_')]
print(f"Number of EEG features: {len(eeg_cols)}")
print(f"Number of EOG features: {len(eog_cols)}")
print(f"Labels column present: {'label' in sample_data.columns}")

feature_stats = defaultdict(lambda: {'min': float('inf'), 'max': float('-inf'), 'sum': 0, 'count': 0, 'missing': 0})
label_counts = defaultdict(int)
total_rows = 0

max_files_to_process = 20
files_to_process = csv_files[:max_files_to_process] if len(csv_files) > max_files_to_process else csv_files

print(f"\nProcessing {len(files_to_process)} out of {len(csv_files)} files for statistics...")
for file in files_to_process:
    data = pd.read_csv(file)
    total_rows += len(data)
    
    if 'label' in data.columns:
        label_vals = data['label'].value_counts().to_dict()
        for k, v in label_vals.items():
            label_counts[k] += v
    
    for col in data.columns:
        if col == 'label':
            continue
        
        if pd.api.types.is_numeric_dtype(data[col]):
            feature_stats[col]['min'] = min(feature_stats[col]['min'], data[col].min())
            feature_stats[col]['max'] = max(feature_stats[col]['max'], data[col].max())
            feature_stats[col]['sum'] += data[col].sum()
            feature_stats[col]['count'] += data[col].count()
            feature_stats[col]['missing'] += data[col].isna().sum()
        else:
            if 'is_numeric' not in feature_stats[col]:
                feature_stats[col]['is_numeric'] = False
                feature_stats[col]['min'] = "N/A"
                feature_stats[col]['max'] = "N/A"
                feature_stats[col]['sum'] = "N/A"
            feature_stats[col]['count'] += data[col].count()
            feature_stats[col]['missing'] += data[col].isna().sum()

for feature, stats in feature_stats.items():
    if stats.get('is_numeric', True) and stats['count'] > 0 and stats['sum'] != "N/A":
        stats['mean'] = stats['sum'] / stats['count']
    else:
        stats['mean'] = "N/A"

stats_df = pd.DataFrame.from_dict(feature_stats, orient='index')
print("\nFeature statistics summary:")
numeric_stats = stats_df[stats_df.get('is_numeric', True) != False]
if not numeric_stats.empty:
    print(f"Min value across all numeric features: {numeric_stats['min'].min()}")
    print(f"Max value across all numeric features: {numeric_stats['max'].max()}")
    numeric_means = pd.to_numeric(numeric_stats['mean'], errors='coerce')
    valid_means = numeric_means.dropna()
    if not valid_means.empty:
        print(f"Average of means across all numeric features: {valid_means.mean()}")
print(f"Total missing values: {stats_df['missing'].sum()}")

if 'label' in sample_data.columns:
    print("\nLabel distribution:")
    labels_series = pd.Series(label_counts).sort_index()
    print(labels_series)
    
    plt.figure(figsize=(10, 6))
    plt.bar(labels_series.index.astype(str), labels_series.values, alpha=0.7)
    plt.title('Label Distribution')
    plt.xlabel('Sleep Stage')
    plt.ylabel('Count')
    plt.xticks(rotation=0)
    plt.grid(alpha=0.3)
    plt.savefig('label_distribution.png')
    plt.close()

print("\nGenerating visualizations...")

# NEW: Distribution visualizations for each feature type
print("Generating feature distribution plots...")

# Create distribution plots for top EEG features
if len(eeg_cols) > 0:
    plt.figure(figsize=(15, 10))
    selected_eeg = eeg_cols[:5]  # Select first 5 EEG features
    
    for i, col in enumerate(selected_eeg):
        plt.subplot(2, 3, i+1)
        sns.histplot(sample_data[col], kde=True)
        plt.title(f'Distribution of {col}')
        plt.tight_layout()
    
    plt.savefig('eeg_feature_distributions.png')
    plt.close()

# Create distribution plots for top EOG features
if len(eog_cols) > 0:
    plt.figure(figsize=(15, 10))
    selected_eog = eog_cols[:5]  # Select first 5 EOG features
    
    for i, col in enumerate(selected_eog):
        plt.subplot(2, 3, i+1)
        sns.histplot(sample_data[col], kde=True)
        plt.title(f'Distribution of {col}')
        plt.tight_layout()
    
    plt.savefig('eog_feature_distributions.png')
    plt.close()

# NEW: Box plots of features by sleep stage
if 'label' in sample_data.columns:
    print("Generating feature boxplots by sleep stage...")
    
    # Select top 3 EEG and EOG features based on feature importance (if available)
    if 'label' in sample_data.columns:
        X = sample_data.drop('label', axis=1)
        y = sample_data['label']
        
        scaler = StandardScaler()
        X_scaled = scaler.fit_transform(X)
        
        rf = RandomForestClassifier(n_estimators=30, random_state=42)
        rf.fit(X_scaled, y)
        
        importances = pd.Series(rf.feature_importances_, index=X.columns)
        top_eeg = [col for col in importances.sort_values(ascending=False).index if col.startswith('eeg_')][:3]
        top_eog = [col for col in importances.sort_values(ascending=False).index if col.startswith('eog_')][:3]
    else:
        # If no label for importance ranking, just take first features
        top_eeg = eeg_cols[:3]
        top_eog = eog_cols[:3]
    
    # Create boxplots of top EEG features by sleep stage
    if len(top_eeg) > 0:
        plt.figure(figsize=(15, 12))
        
        for i, feature in enumerate(top_eeg):
            plt.subplot(len(top_eeg), 1, i+1)
            sns.boxplot(x='label', y=feature, data=sample_data)
            plt.title(f'{feature} by Sleep Stage')
            plt.tight_layout()
        
        plt.savefig('eeg_boxplots_by_stage.png')
        plt.close()
    
    # Create boxplots of top EOG features by sleep stage
    if len(top_eog) > 0:
        plt.figure(figsize=(15, 12))
        
        for i, feature in enumerate(top_eog):
            plt.subplot(len(top_eog), 1, i+1)
            sns.boxplot(x='label', y=feature, data=sample_data)
            plt.title(f'{feature} by Sleep Stage')
            plt.tight_layout()
        
        plt.savefig('eog_boxplots_by_stage.png')
        plt.close()
    
    # NEW: Violin plots for top 2 features
    plt.figure(figsize=(14, 10))
    if len(top_eeg) > 0 and len(top_eog) > 0:
        combined_top = top_eeg[:1] + top_eog[:1]
        
        for i, feature in enumerate(combined_top):
            plt.subplot(1, 2, i+1)
            sns.violinplot(x='label', y=feature, data=sample_data, inner='quartile')
            plt.title(f'{feature} Distribution by Sleep Stage')
        
        plt.tight_layout()
        plt.savefig('top_features_violin_plots.png')
        plt.close()

# Compare EEG vs EOG distributions for common features
if len(eeg_cols) > 0 and len(eog_cols) > 0:
    eeg_feature_names = [col.replace('eeg_', '') for col in eeg_cols]
    eog_feature_names = [col.replace('eog_', '') for col in eog_cols]
    
    common_features = set(eeg_feature_names) & set(eog_feature_names)
    
    if len(common_features) > 0:
        features_to_compare = list(common_features)[:5]
        
        fig, axes = plt.subplots(len(features_to_compare), 1, figsize=(12, 4*len(features_to_compare)))
        
        data = pd.read_csv(sample_file)
        
        for i, feature in enumerate(features_to_compare):
            eeg_data = data[f'eeg_{feature}']
            eog_data = data[f'eog_{feature}']
            
            ax = axes[i] if len(features_to_compare) > 1 else axes
            sns.kdeplot(data=eeg_data, ax=ax, label='EEG')
            sns.kdeplot(data=eog_data, ax=ax, label='EOG')
            ax.set_title(f'Distribution of {feature}')
            ax.legend()
        
        plt.tight_layout()
        plt.savefig('eeg_vs_eog_distributions.png')
        plt.close()

# NEW: Plot the feature value distributions for different sleep stages
if 'label' in sample_data.columns and len(common_features) > 0:
    # Select a feature that appears in both EEG and EOG
    feature_to_plot = list(common_features)[0]
    
    plt.figure(figsize=(12, 8))
    
    # EEG feature across sleep stages
    plt.subplot(1, 2, 1)
    for stage in sorted(sample_data['label'].unique()):
        stage_data = sample_data[sample_data['label'] == stage][f'eeg_{feature_to_plot}']
        sns.kdeplot(data=stage_data, label=f'Stage {stage}')
    
    plt.title(f'EEG {feature_to_plot} Distribution by Sleep Stage')
    plt.legend()
    
    # EOG feature across sleep stages
    plt.subplot(1, 2, 2)
    for stage in sorted(sample_data['label'].unique()):
        stage_data = sample_data[sample_data['label'] == stage][f'eog_{feature_to_plot}']
        sns.kdeplot(data=stage_data, label=f'Stage {stage}')
    
    plt.title(f'EOG {feature_to_plot} Distribution by Sleep Stage')
    plt.legend()
    
    plt.tight_layout()
    plt.savefig('feature_distribution_by_stage.png')
    plt.close()

# Correlation matrix
plt.figure(figsize=(14, 12))
correlation_cols = eeg_cols[:10] + eog_cols[:10]
correlation_matrix = sample_data[correlation_cols].corr()
sns.heatmap(correlation_matrix, annot=False, cmap='coolwarm', vmin=-1, vmax=1)
plt.title('Feature Correlation Matrix')
plt.xticks(rotation=90)
plt.yticks(rotation=0)
plt.tight_layout()
plt.savefig('feature_correlation_matrix.png')
plt.close()

# NEW: Create a more focused correlation matrix showing just the top correlated features
if len(correlation_cols) > 8:
    corr_values = correlation_matrix.abs().unstack()
    # Remove self-correlations
    corr_values = corr_values[corr_values < 1]
    top_corr = corr_values.sort_values(ascending=False)[:20].index
    
    # Get unique column names from the top correlations
    top_cols = list(set([col[0] for col in top_corr] + [col[1] for col in top_corr]))[:10]
    
    plt.figure(figsize=(12, 10))
    sns.heatmap(sample_data[top_cols].corr(), annot=True, cmap='coolwarm', vmin=-1, vmax=1)
    plt.title('Top Correlated Features')
    plt.xticks(rotation=90)
    plt.yticks(rotation=0)
    plt.tight_layout()
    plt.savefig('top_correlated_features.png')
    plt.close()

# Feature importance for sleep stage classification
if 'label' in sample_data.columns:
    X = sample_data.drop('label', axis=1)
    y = sample_data['label']
    
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)
    
    rf = RandomForestClassifier(n_estimators=50, random_state=42)
    rf.fit(X_scaled, y)
    
    importances = pd.Series(rf.feature_importances_, index=X.columns)
    importances = importances.sort_values(ascending=False)
    
    plt.figure(figsize=(12, 8))
    importances[:20].plot(kind='bar')
    plt.title('Top 20 Feature Importance for Sleep Stage Classification')
    plt.ylabel('Importance')
    plt.tight_layout()
    plt.savefig('feature_importance.png')
    plt.close()
    
    # NEW: Grouped feature importance by source (EEG vs EOG)
    eeg_importance = importances[importances.index.str.startswith('eeg_')].sum()
    eog_importance = importances[importances.index.str.startswith('eog_')].sum()
    
    plt.figure(figsize=(8, 6))
    plt.bar(['EEG Features', 'EOG Features'], [eeg_importance, eog_importance])
    plt.title('Importance of EEG vs EOG Features')
    plt.ylabel('Total Importance')
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig('eeg_vs_eog_importance.png')
    plt.close()

# PCA visualization
if 'label' in sample_data.columns:
    X = sample_data.drop('label', axis=1)
    y = sample_data['label']
    
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)
    
    pca = PCA(n_components=2)
    X_pca = pca.fit_transform(X_scaled)
    
    pca_df = pd.DataFrame({
        'PC1': X_pca[:, 0],
        'PC2': X_pca[:, 1],
        'Label': y
    })
    
    plt.figure(figsize=(10, 8))
    sns.scatterplot(data=pca_df, x='PC1', y='PC2', hue='Label', palette='viridis', alpha=0.7)
    plt.title('PCA of catch22 Features by Sleep Stage')
    plt.tight_layout()
    plt.savefig('pca_visualization.png')
    plt.close()
    
    print(f"\nVariance explained by PCA components:")
    print(f"PC1: {pca.explained_variance_ratio_[0]*100:.2f}%")
    print(f"PC2: {pca.explained_variance_ratio_[1]*100:.2f}%")
    print(f"Total: {sum(pca.explained_variance_ratio_)*100:.2f}%")
    
    # NEW: PCA with 3 components to see if we can get better separation
    pca3 = PCA(n_components=3)
    X_pca3 = pca3.fit_transform(X_scaled)
    
    pca3_df = pd.DataFrame({
        'PC1': X_pca3[:, 0],
        'PC2': X_pca3[:, 1],
        'PC3': X_pca3[:, 2],
        'Label': y
    })
    
    # Create pair plot for 3D PCA
    sns.pairplot(pca3_df, hue='Label', vars=['PC1', 'PC2', 'PC3'], palette='viridis')
    plt.suptitle('PCA Components Pairplot by Sleep Stage', y=1.02)
    plt.tight_layout()
    # plt.savefig('pca_pairplot.png')
    plt.show()
    plt.close()
    
    print(f"PC3: {pca3.explained_variance_ratio_[2]*100:.2f}%")
    print(f"Total with 3 components: {sum(pca3.explained_variance_ratio_)*100:.2f}%")

# NEW: Create plots that show feature stability across recordings
if len(files_to_process) > 2:
    print("Analyzing feature stability across recordings...")
    
    # Select a few representative features
    if len(eeg_cols) > 0 and len(eog_cols) > 0:
        features_to_track = eeg_cols[:2] + eog_cols[:2]
        
        # Store mean values for each feature across files
        feature_means = {feature: [] for feature in features_to_track}
        file_names = []
        
        for file in files_to_process[:10]:  # Limit to first 10 files
            data = pd.read_csv(file)
            file_name = os.path.basename(file).replace("_catch22_features.csv", "")
            file_names.append(file_name)
            
            for feature in features_to_track:
                if feature in data.columns and pd.api.types.is_numeric_dtype(data[feature]):
                    feature_means[feature].append(data[feature].mean())
                else:
                    feature_means[feature].append(np.nan)
        
        # Create plot
        plt.figure(figsize=(14, 8))
        for feature in features_to_track:
            plt.plot(feature_means[feature], marker='o', label=feature)
        
        plt.title('Feature Stability Across Recordings')
        plt.xlabel('Recording')
        plt.ylabel('Mean Feature Value')
        plt.xticks(range(len(file_names)), file_names, rotation=90)
        plt.legend()
        plt.grid(alpha=0.3)
        plt.tight_layout()
        # plt.savefig('feature_stability.png')
        plt.show()
        plt.close()

print("\nExploratory analysis complete! Generated visualizations have been saved.")

Step 1: Extracting tarfile...
Extraction complete!
Found 179 CSV files and 178 NPZ files
Found mapping file with 178 recordings
Number of unique subjects: 94

Distribution of recordings per subject:
Min: 1, Max: 2, Mean: 1.89

Subject distribution:
SC (Sleep Cassette) subjects: 72
ST (Sleep Telemetry) subjects: 22

Sample data structure from SC4032E0_c22.csv:
Shape: (2732, 45)
Columns: 45
Number of EEG features: 22
Number of EOG features: 22
Labels column present: True

Processing 20 out of 179 files for statistics...

Feature statistics summary:
Min value across all numeric features: -3.026237183749113
Max value across all numeric features: 1917.0
Average of means across all numeric features: 15.894723879660908
Total missing values: 0

Label distribution:
0    29459
1     2095
2     8337
3     1754
4     3281
dtype: int64

Generating visualizations...
Generating feature distribution plots...
Generating feature boxplots by sleep stage...

Variance explained by PCA components:
PC1: 24.8